# 13.3 视频语言 (Video-Language)

视频语言模型在视觉语言基础上增加时序维度，理解视频中的动态信息。

本节涵盖：视频编码器、视频语言模型、时序建模

## 1. 视频编码器与时序建模

**视频编码器**：从视频中提取时空特征。
- 帧采样：均匀采样/关键帧采样
- 每帧用ViT编码，再通过时序建模聚合

**时序建模**：捕获帧间的时间关系。
- 时序注意力：帧间的自注意力
- 时序池化：平均/最大池化聚合时间维度
- 时序Transformer：专门处理时间序列

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class SimpleVideoEncoder(nn.Module):
    def __init__(self, d_frame=64, d_video=128, n_frames=8):
        super().__init__()
        self.frame_encoder = nn.Sequential(
            nn.Linear(3 * 16 * 16, 128), nn.ReLU(), nn.Linear(128, d_frame)
        )
        self.temporal_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_frame, 2, d_frame * 4, batch_first=True), 1
        )
        self.proj = nn.Linear(d_frame, d_video)

    def forward(self, frames):
        B, T, C, H, W = frames.shape
        frame_features = self.frame_encoder(frames.reshape(B, T, -1))
        temporal_features = self.temporal_encoder(frame_features)
        return self.proj(temporal_features)

video_enc = SimpleVideoEncoder()
frames = torch.randn(2, 8, 3, 16, 16)
video_features = video_enc(frames)

print('=== Video-Language Model ===')
print(f'Input: {frames.shape} (batch, frames, channels, height, width)')
print(f'Frame features: {frames.shape[1]} frames encoded')
print(f'Video features: {video_features.shape}')
print(f'\nKey: Frame encoder -> Temporal Transformer -> Video tokens')
print(f'Temporal modeling captures motion and event sequences across frames.')